<img src="https://th.bing.com/th/id/R.0b9618782d4e7062573f5983d876649a?rik=7HWDbD%2feXNOe1A&pid=ImgRaw&r=0" width=150>


**I758 Wissens- und KI-basierte Systeme**

# Explorative Datenanalyse - Teil 1: Datenqualität erkunden & korrigieren
(c) Ricardo Knauer, Raphael Wallsberger, Christina Kratsch

In der Realität sind Datensätze selten so gutartig wie unsere Maschinendaten aus der letzten Übung. Regelmäßig kommt es im Alltag zu "unerwarteten" Datenwerten. Ein guter Data Scientist verwendet deshalb einen wesentlichen Anteil seiner Zeit damit, die Qualität der Daten zu überprüfen und ggf. für die gestellte Aufgabe optimieren. Dies ist eine Wissenschaft für sich (sogenanntes *Data Engineering*), hier erhalten Sie nur einen ersten Einblick. 

Nehmen wir an, wir hätten nur eine deutlich schlechtere Messreihe zur Verfügung. Alles, was wir wissen, ist, dass mit den Daten etwas "nicht in Ordnung" ist. Wir müssen uns langsam vortasten. Die schlechte Messreihe finden Sie in einer anderen Datei:

In [119]:
import pandas as pd

df = pd.read_csv("data/machine_data_broken.csv", sep=";")
df.head()

,Maschine,Mode,Produkt,Strom / A,Drehmoment / Nm,Drehzahl / 1/min,Temp Umgebung / degC,Temp Umrichter / degC,Temp Werkzeug / degC,Bearbeitungszeit / s
0,A,2.0,X,NaN,"48,64",1463,"21,5","23,3","95,1","21,4"
1,B,2.0,Y,NaN,"50,92",1462,"20,9","22,8","98,2","24,8"
2,C,2.0,X,NaN,"46,85",1462,"21,3","23,8","93,1","21,4"
3,B,2.0,Y,NaN,"49,69",1463,"21,5","24,4","97,7",NaN
4,C,3.0,X,"26,195","53,29",1462,NaN,"25,3","104,1","19,9"


Der Datensatz ist deutlich kleiner - aber immerhin stimmt das Format:

In [120]:
df.shape

(299, 10)

Mit ```.info()``` bekommen Sie einen Einblick in die Datentypen jeder Spalte:

In [121]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Maschine               299 non-null    str    
 1   Mode                   298 non-null    float64
 2   Produkt                299 non-null    str    
 3   Strom / A              285 non-null    str    
 4   Drehmoment / Nm        299 non-null    str    
 5   Drehzahl / 1/min       299 non-null    int64  
 6   Temp Umgebung / degC   298 non-null    str    
 7   Temp Umrichter / degC  298 non-null    str    
 8   Temp Werkzeug / degC   298 non-null    str    
 9   Bearbeitungszeit / s   245 non-null    str    
dtypes: float64(1), int64(1), str(8)
memory usage: 23.5 KB


Hier zeigt sich das erste Problem: einige Werte scheinen in den Spalten zu fehlen, der ```Non-Null Count``` entspricht nämlich nicht für jede Spalte der Zeilenzahl.
Außerdem erkennen wir einen Fehler beim Einlesen der Daten: die meisten Spalten sind vom Typ ```object```. Soll heißen: Pandas erkennt nicht, dass es sich um numerische Werte handelt. Dies lässt sich einfach erklären - wir haben beim Einlesen nicht erwähnt, dass die Zahlen nicht mit der "deutschen" Schreibweise mit Komma als Trennzeichen notiert sind. Eine leidige und typische Fehlerquelle im Umgang mit Daten (die sich leider meist erst durch sehr kryptische Fehler weiter unten in den Pipeline bemerkbar macht).

Also nochmal - lassen Sie uns ```df``` nochmal überschreiben:

In [122]:
df = pd.read_csv("data/machine_data_broken.csv", sep=";", decimal=",")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Maschine               299 non-null    str    
 1   Mode                   298 non-null    float64
 2   Produkt                299 non-null    str    
 3   Strom / A              285 non-null    float64
 4   Drehmoment / Nm        299 non-null    float64
 5   Drehzahl / 1/min       299 non-null    int64  
 6   Temp Umgebung / degC   298 non-null    float64
 7   Temp Umrichter / degC  298 non-null    float64
 8   Temp Werkzeug / degC   298 non-null    float64
 9   Bearbeitungszeit / s   245 non-null    float64
dtypes: float64(7), int64(1), str(2)
memory usage: 23.5 KB


Viel besser! Aber da ist noch das Problem mit den fehlenden Werten. die letzte Spalte (Bearbeitungszeit) und auch einige andere Spalten scheinen unter Datenfehlern zu leiden.

<span style="color:#FF5F00"><b>AUFGABE 1:</b></span><br>
Wir können es uns einfach machen. Nutzen Sie die  eingebauten Funktionen ```.isnull``` und ```.sum```, um die Nullwerte zu zählen:

In [123]:
df.isnull().sum()

Maschine                  0
Mode                      1
Produkt                   0
Strom / A                14
Drehmoment / Nm           0
Drehzahl / 1/min          0
Temp Umgebung / degC      1
Temp Umrichter / degC     1
Temp Werkzeug / degC      1
Bearbeitungszeit / s     54
dtype: int64

Die Spalte *Bearbeitungszeit / s* scheint massiv von unserem Problem betroffen. Hier fehlen so viele Werte, dass wir davon ausgehen können, dass die Spalte praktisch wertlos ist. 

<span style="color:#FF5F00"><b>AUFGABE 2:</b></span><br>
Entfernen Sie mit `del` die Spalte aus dem Data Frame!

In [124]:
del df["Bearbeitungszeit / s"]

Gut zu wissen: für komplizierte Verfahren gibt es die mächtige Funktion ```.dropna```, die Spalten anhand fester Qualitätskriterien entfernen kann. Die wollen wir hier aber heute **nicht** einsetzen. Führen Sie den auskommentierten Code NICHT aus.

In [125]:
# Drop any rows which have any NaNs 
#df.dropna(axis=0) 
# Drop columns with over 70% non-NaNs 
#df.dropna(thresh=int(df.shape[0] * .7), axis=1)

# Löschen der Bearbeitungszeit mit DropNa + 90% Kriterium (min. 90% der Spalten müssen Wert haben, Spalte drin bleibt)
# df = df.dropna(thresh=int(df.shape[0] * .9), axis=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Maschine               299 non-null    str    
 1   Mode                   298 non-null    float64
 2   Produkt                299 non-null    str    
 3   Strom / A              285 non-null    float64
 4   Drehmoment / Nm        299 non-null    float64
 5   Drehzahl / 1/min       299 non-null    int64  
 6   Temp Umgebung / degC   298 non-null    float64
 7   Temp Umrichter / degC  298 non-null    float64
 8   Temp Werkzeug / degC   298 non-null    float64
dtypes: float64(6), int64(1), str(2)
memory usage: 21.2 KB


Die Spalte *Strom / A* ist uns wichtig - aber auch hier fehlen massiv Werte. Schauen wir uns die Spalte nochmal an:

In [126]:
df["Strom / A"]

0         NaN
1         NaN
2         NaN
3         NaN
4      26.195
        ...  
294    22.472
295    20.123
296    23.154
297    22.347
298    23.761
Name: Strom / A, Length: 299, dtype: float64

Pandas bietet die Funktion ```.fillna```, mit der leere Werte gefunden und neu gesetzt werden können. Zum Beispiel auf 0:

In [127]:
df['Strom / A'].fillna(0.0)

0       0.000
1       0.000
2       0.000
3       0.000
4      26.195
        ...  
294    22.472
295    20.123
296    23.154
297    22.347
298    23.761
Name: Strom / A, Length: 299, dtype: float64

<span style="color:#FF5F00"><b>AUFGABE 3:</b></span><br>
Achtung: achten Sie nochmal auf den Code der letzten Zeile, und dann schauen Sie sich nochmal die Werte in ```df``` an. Fällt Ihnen etwas auf? Wie hat sich ```df``` verändert?

In [128]:
# Mit Kopie arbeiten, um spätere Aufgaben nicht zu stören
df_copy = df.copy()
df_copy

,Maschine,Mode,Produkt,Strom / A,Drehmoment / Nm,Drehzahl / 1/min,Temp Umgebung / degC,Temp Umrichter / degC,Temp Werkzeug / degC
0,A,2.0,X,NaN,48.64,1463,21.5,23.3,95.1
1,B,2.0,Y,NaN,50.92,1462,20.9,22.8,98.2
2,C,2.0,X,NaN,46.85,1462,21.3,23.8,93.1
3,B,2.0,Y,NaN,49.69,1463,21.5,24.4,97.7
4,C,3.0,X,26.195,53.29,1462,NaN,25.3,104.1
...,...,...,...,...,...,...,...,...,...
294,B,2.0,Y,22.472,50.24,1461,20.6,23.9,97.5
295,A,1.0,Y,20.123,45.67,1463,21.3,22.6,90.0
296,C,2.0,X,23.154,47.19,1460,20.6,24.5,93.8
297,B,2.0,Y,22.347,49.99,1460,19.6,23.8,97.5


<span style="color:green"><b>schreiben sie nachfolgend Ihre schriftliche Antwort zu AUFGABE 3:</b></span><br>

Der Inhalt der *Strom / A* Spalte hat sich bei dem Funktionsaufruf nicht geändert. Das liegt daran, dass hier nicht in-place gearbeitet wurde. Die Funktion gibt eine `Series` zurück, der aber von uns nicht genutzt oder gespeichert wird.

`inplace=True` funktioniert nicht wie gedacht für spaltenbezogene Funktionen, (Erklärung: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html), deshalb muss die Spalte manuell überschrieben werden:

In [129]:
# Vorher
df_copy.isnull().sum()

Maschine                  0
Mode                      1
Produkt                   0
Strom / A                14
Drehmoment / Nm           0
Drehzahl / 1/min          0
Temp Umgebung / degC      1
Temp Umrichter / degC     1
Temp Werkzeug / degC      1
dtype: int64

In [130]:
df_copy['Strom / A'] = df_copy['Strom / A'].fillna(0.0)
df_copy

,Maschine,Mode,Produkt,Strom / A,Drehmoment / Nm,Drehzahl / 1/min,Temp Umgebung / degC,Temp Umrichter / degC,Temp Werkzeug / degC
0,A,2.0,X,0.000,48.64,1463,21.5,23.3,95.1
1,B,2.0,Y,0.000,50.92,1462,20.9,22.8,98.2
2,C,2.0,X,0.000,46.85,1462,21.3,23.8,93.1
3,B,2.0,Y,0.000,49.69,1463,21.5,24.4,97.7
4,C,3.0,X,26.195,53.29,1462,NaN,25.3,104.1
...,...,...,...,...,...,...,...,...,...
294,B,2.0,Y,22.472,50.24,1461,20.6,23.9,97.5
295,A,1.0,Y,20.123,45.67,1463,21.3,22.6,90.0
296,C,2.0,X,23.154,47.19,1460,20.6,24.5,93.8
297,B,2.0,Y,22.347,49.99,1460,19.6,23.8,97.5


Nun sind alle *NaN* Zellen in der Spalte beseitigt. Die anderen Spalten sind unberührt geblieben.
Der Unterschied ist schnell mit `df.isnull().sum()` gezeigt:

In [131]:
# Nacher
df_copy.isnull().sum()

Maschine                 0
Mode                     1
Produkt                  0
Strom / A                0
Drehmoment / Nm          0
Drehzahl / 1/min         0
Temp Umgebung / degC     1
Temp Umrichter / degC    1
Temp Werkzeug / degC     1
dtype: int64

<span style="color:#FF5F00"><b>AUFGABE 4:</b></span><br>
Noch mehr Aufgaben für Sie: wie sicher fühlen Sie sich mit Data Science? Versuchen Sie doch mal, die fehlenden Werte nicht durch 0.0, sondern durch etwas intelligenteres zu ersetzen, zum Beispiel den Mittelwert oder den Median. Begründen Sie nachfolgend Ihre Entscheidung für welche Ersatzwertbestimmung sie sich für fehlende Werte entschieden haben.  
Schreiben sie die Spalte jetzt auch wieder in den Data Frame zurück, damit sich `df` tatsächlich ändert:

In [132]:
# aus der Vorlesung 04 - Datenqualität
missing_col = ["Strom / A"]

for i in missing_col:
    df.loc[df.loc[:,i].isnull(),i]=df.loc[:,i].median()

df

,Maschine,Mode,Produkt,Strom / A,Drehmoment / Nm,Drehzahl / 1/min,Temp Umgebung / degC,Temp Umrichter / degC,Temp Werkzeug / degC
0,A,2.0,X,22.168,48.64,1463,21.5,23.3,95.1
1,B,2.0,Y,22.168,50.92,1462,20.9,22.8,98.2
2,C,2.0,X,22.168,46.85,1462,21.3,23.8,93.1
3,B,2.0,Y,22.168,49.69,1463,21.5,24.4,97.7
4,C,3.0,X,26.195,53.29,1462,NaN,25.3,104.1
...,...,...,...,...,...,...,...,...,...
294,B,2.0,Y,22.472,50.24,1461,20.6,23.9,97.5
295,A,1.0,Y,20.123,45.67,1463,21.3,22.6,90.0
296,C,2.0,X,23.154,47.19,1460,20.6,24.5,93.8
297,B,2.0,Y,22.347,49.99,1460,19.6,23.8,97.5


<span style="color:green"><b>schreiben sie nachfolgend Ihre schriftliche Antwort zu AUFGABE 4:</b></span><br>

Ich habe mich für den Median entschieden, da es einfach ist, ihn anzuwenden und er im Gegenteil zum Durchschnitt (*mean*) nicht so anfällig für Ausreißer ist.

<span style="color:#FF5F00"><b>AUFGABE 5:</b></span><br>
Plotten Sie nun zum Abschluss ihre Daten nochmal, zum Beispiel als Scatter Plot zwischen Strom und Drehmoment und betrachten Sie das Ergebnis Ihrer Arbeit. Vielleicht fällt Ihnen ja noch ein drittes Problem in unserem Datensatz auf? Welche Ursachen kann dies haben? Schauen Sie mal in die "Ecken" des Plots!

In [133]:
# schreiben Sie hier Ihren Code
pd.options.plotting.backend = "plotly"

df.plot(kind="scatter", y="Strom / A", x = "Drehmoment / Nm", color="Maschine")


<span style="color:green"><b>schreiben sie nachfolgend Ihre schriftliche Antwort zu AUFGABE 5:</b></span><br>

# Explorative Datenanalyse - Teil 2: Daten skalieren und verarbeiten mit `scikit learn`
(c) Christina Kratsch

Viele Machine Learning Algorithmen machen Annahmen über ihre Daten, z.B. über die zugrundeliegende statistische Verteilung, den Wertebereich oder den Datentyp. Die Data Science Bibliothek `scikit learn` bietet neben einer extrem umfangreichen Sammlung an Algorithmen (und einer exzellenten Dokumentation) auch mit `sklearn.preprocessing` eine Bibliothek zur Vorverarbeitung von Daten.

## Skalieren und Normalisieren

Zuerst importieren wir die notwendigen Bibliotheken und laden unsere Daten.

In [134]:
import numpy as np
from sklearn import preprocessing

# Angenommen, dies sind unsere Daten
X_train = np.array([[1., -1., 2.],
                    [2., 0., 0.],
                    [0., 1., -1.]])

Die Standardisierung von Datensätzen ist eine gängige Anforderung für viele Machine-Learning-Schätzer. Sie könnten sich schlecht verhalten, wenn die einzelnen Merkmale nicht mehr oder weniger wie standardnormalverteilte Daten aussehen: Gaussian mit Null-Mittelwert und Einheitsvarianz.

<span style="color:#FF5F00"><b>AUFGABE 6:</b></span><br>
Überlegen Sie einen Moment, was ein Grund sein könnte, warum die Syntax des `StandardScaler` so umständlich ist. Schauen Sie ggf. auch in der [Dokumentation](https://scikit-learn.org/stable/modules/preprocessing.html) nach!

In [135]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)
print(X_scaled)

[[ 0.         -1.22474487  1.33630621]
 [ 1.22474487  0.         -0.26726124]
 [-1.22474487  1.22474487 -1.06904497]]


<span style="color:green"><b>schreiben sie nachfolgend Ihre Antwort zu AUFGABE 6:</b></span><br>

Der Grund für die realtiv komplizierte Syntax des `StandardScaler`s sind die vielen Optionen und Anwendungsfälle, die er bietet. 

<span style="color:#FF5F00"><b>AUFGABE 7:</b></span><br>
Legen Sie eine Kopie `df_num` Ihres Data Frames an. Nutzen Sie den StandardScaler, um alle Werte darin zu normalisieren. Funktioniert das? Was fällt Ihnen auf? Korrigieren Sie ggf. `df_num` entsprechend. Wie gehen Sie mit der Spalte `mode` um?

In [136]:
df_num = df.copy()

# drop categorical columns
del df_num["Maschine"], df_num["Produkt"], df_num["Mode"]

df_num_scaled = scaler.fit_transform(df_num)
print(df_num_scaled)

[[-0.16226956 -0.06146717  1.08473773  0.40110056 -0.43561318 -0.06316182]
 [-0.16226956  0.26113966  0.57235766 -0.03807139 -0.89758018  0.46250383]
 [-0.16226956 -0.31474184  0.57235766  0.25470991  0.02635382 -0.40230094]
 ...
 [ 0.22611557 -0.2666338  -0.45240246 -0.25765737  0.67310763 -0.28360225]
 [-0.09176151  0.12955003 -0.45240246 -0.98961064  0.02635382  0.34380513]
 [ 0.46521271 -0.09118096 -0.45240246 -0.98961064 -0.89758018 -0.24968834]]


<span style="color:green"><b>schreiben sie nachfolgend Ihre Antwort zu AUFGABE 7:</b></span><br>

Es funktioniert nicht direkt, den kompletten Dataframe zu standardisieren, da Median und Standardabweichung berechnet werden müssen, um numerische Werte zu standardisieren. Für kategorische Werte lassen sich diese Werte nicht berechnen, da sie nicht geordnet sind und keine festen Abstände zueinander haben.

Aus diesem Grund habe ich die Spalten mit kategorischen Messwerten vor der Standardisierung aus dem Dataframe entfernt.

Manchmal möchte man Daten in einem bestimmten Wertebereich (z.B. [0, 1]) haben, z.B. um sie als Wahrscheinlichkeiten interpretieren zu können. Auch dafür bietet `sklearn.preprocessing` Möglichkeiten:

In [137]:
X_train = np.array([[1., -1., 2.],
                    [2., 0., 0.],
                    [0., 1., -1.]])

min_max_scaler = preprocessing.MinMaxScaler()
X_train_minmax = min_max_scaler.fit_transform(X_train)
X_train_minmax

array([[0.5       , 0.        , 1.        ],
       [1.        , 0.5       , 0.33333333],
       [0.        , 1.        , 0.        ]])

<span style="color:#FF5F00"><b>AUFGABE 8:</b></span><br>Skalieren Sie die Einträge in `df_num` auf den Wertebereich [-3, 3]!

In [138]:
min_max_scaler = preprocessing.MinMaxScaler(feature_range=(-3, 3))
df_num_minmax = min_max_scaler.fit_transform(df_num)
df_num_minmax

array([[-0.8678311 , -2.60818766,  0.81818182, -0.21428571, -0.84375   ,
        -0.55183946],
       [-0.8678311 , -2.48428584,  0.27272727, -0.64285714, -1.3125    ,
         0.07023411],
       [-0.8678311 , -2.70546146,  0.27272727, -0.35714286, -0.375     ,
        -0.95317726],
       ...,
       [-0.33519402, -2.68698487, -0.81818182, -0.85714286,  0.28125   ,
        -0.81270903],
       [-0.77113532, -2.53482474, -0.81818182, -1.57142857, -0.375     ,
        -0.07023411],
       [-0.0072927 , -2.61959967, -0.81818182, -1.57142857, -1.3125    ,
        -0.77257525]], shape=(299, 6))

## Umgang mit kategorischen Variablen

Wir haben noch einige Spalten in unseren Maschinendaten, die wir nicht adressiert haben: die kategorischen Werte wie z.B. die Produkt-Klasse. Manche ML-Verfahren können aber grundsätzlich nur mit numerischen Werten umgehen. 

Die einfachste Möglichkeit, Kategorien in Zahlen umzuwandeln, ist, diese einfach zu "übersetzen", wie es im folgenden Beispiel mit drei Variablen passiert:

In [139]:
enc = preprocessing.OrdinalEncoder()
X = [['male', 'from US', 'uses Safari'], ['female', 'from Europe', 'uses Firefox']]
enc.fit(X)
enc.transform([['female', 'from US', 'uses Safari']])

array([[0., 1., 1.]])

<span style="color:#FF5F00"><b>AUFGABE 9:</b></span><br>
Erweitern Sie das Beispiel um weitere Belegungen (zum Beispiel das Geschlecht _non_binary_ oder die Herkunft _from Korea_) und kodieren Sie eine Beispiel-Belegung.

In [140]:
# schrieben sie hier Ihren Code
X.append(['attack helicopter', 'from Germany', 'uses Chrome'])
X.append(['non-binary', 'from Portugal', 'uses Brave'])

enc.fit(X)
enc.transform([['attack helicopter', 'from Portugal', 'uses Brave']])

array([[0., 2., 0.]])

<span style="color:#FF5F00"><b>AUFGABE 10:</b></span><br>
Was passiert, wenn einer der Werte nicht definiert ist (nutzen Sie `np.nan`)? Was macht die Belegung  `encoded_missing_value`?

In [141]:
# schreiben sie hier Ihren Code
# enc.transform([['attack helicopter', np.nan, 'uses Brave']]) --> ValueError: Found unknown categories [nan] in column 1 during transform

enc = preprocessing.OrdinalEncoder(encoded_missing_value=np.nan, handle_unknown='use_encoded_value', unknown_value=np.nan)
enc.fit(X)
enc.transform([['male', np.nan , 'uses Vivaldi']])

array([[ 2., nan, nan]])

<span style="color:green"><b>schreiben sie nachfolgend Ihre Antwort zu AUFGABE 10:</b></span><br>

Ist einer der Werte nicht definiert (`np.nan`), kann das Verhalten mit den oben verwendeten Belegungen gesteuert werden. 

`encoded_missing_value=np.nan` --> Gibt an, wie fehlende Werte beim Transformieren kodiert werden.

`handle_unknown='use_encoded_value'` --> Gibt an, wie mit nach `fit(X)` unbekannten Kategorien umgegangen werden soll, in diesem Fall wird `unknown_value` eingesetzt. Standardmäßig wird eigentlich der oben kommentierte `Value Error` geworfen.

`unknown_value=np.nan` --> Gibt an, wie unbekannte Kategorien beim transformieren kodiert werden.

Der fehlende Wert `np.nan` bleibt also gleich, die unbekannte Kategorie `'uses Vivaldi'` wird ebenfalls zu `np.nan` kodiert. Um die beiden Fälle sinnvoll unterscheiden zu können, könnte man statt `np.nan` auch unterschiedliche, negative Kodierungen verwenden. Beim Fitting werden Kategorien standardmäßig nur als positive Floats kodiert, deshalb bieten sich negative Floats für "Fehler"-Kodierung an.

<span style="color:#FF5F00"><b>AUFGABE 11:</b></span><br>Zum Schluss wird es nochmal kniffelig. Informieren Sie sich, was ein [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html#sklearn.preprocessing.OneHotEncoder) macht und kodieren Sie eine kategorische Spalte unseres Maschinendatensatzes (`df`). Achten Sie, wenn möglich, auch darauf, was mit fehlenden Werten passiert!

In [142]:
df_mode = df.copy()
df_mode["Mode"] = df_mode["Mode"].fillna("missing").astype(str)

encoder = preprocessing.OneHotEncoder(sparse_output=False, handle_unknown="ignore")
mode_encoded = encoder.fit_transform(df_mode[["Mode"]])

mode_encoded_df = pd.DataFrame(
    mode_encoded,
    columns=encoder.get_feature_names_out(["Mode"]),
    index=df_mode.index
)

df_mode = pd.concat([df_mode.drop(columns=["Mode"]), mode_encoded_df], axis=1)
df_mode

,Maschine,Produkt,Strom / A,Drehmoment / Nm,Drehzahl / 1/min,Temp Umgebung / degC,Temp Umrichter / degC,Temp Werkzeug / degC,0,Mode_1.0,Mode_2.0,Mode_3.0,Mode_nan
0,A,X,22.168,48.64,1463.0,21.5,23.3,95.1,NaN,0.0,1.0,0.0,0.0
1,B,Y,22.168,50.92,1462.0,20.9,22.8,98.2,NaN,0.0,1.0,0.0,0.0
2,C,X,22.168,46.85,1462.0,21.3,23.8,93.1,NaN,0.0,1.0,0.0,0.0
3,B,Y,22.168,49.69,1463.0,21.5,24.4,97.7,NaN,0.0,1.0,0.0,0.0
4,C,X,26.195,53.29,1462.0,NaN,25.3,104.1,NaN,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,A,Y,20.123,45.67,1463.0,21.3,22.6,90.0,NaN,1.0,0.0,0.0,0.0
296,C,X,23.154,47.19,1460.0,20.6,24.5,93.8,NaN,0.0,1.0,0.0,0.0
297,B,Y,22.347,49.99,1460.0,19.6,23.8,97.5,NaN,0.0,1.0,0.0,0.0
298,C,X,23.761,48.43,1460.0,19.6,22.8,94.0,NaN,0.0,1.0,0.0,0.0


<span style="color:green"><b>schreiben sie nachfolgend Ihre Antwort zu AUFGABE 11:</b></span><br>

`OneHotEncoder` wandelt eine kategoriale Spalte in mehrere binäre Spalten um. Jede Kategorie erhält ihre eigene Spalte, die für passende Zeilen den Wert `1` und sonst `0` enthält. Für die Spalte `Mode` entstehen dadurch eigene Spalten pro Modus. Fehlende Werte habe ich vor dem Encoding durch die Kategorie `"missing"` ersetzt, sodass sie nicht verloren gehen, sondern als eigene Ausprägung mitkodiert werden.

Klasse! Sie haben jetzt einen grundlegenden Überblick über die Möglichkeiten zur Manipulation von Daten gewonnen. Nicht alle Methoden erscheinen Ihnen vielleicht jetzt bereits umfassend sinnvoll - wenn wir weiter voranschreiten, werden Sie aber viele ML-Algorithmen kennenlernen, die auf den hier genannten Verfahren aufbauen. Wenn die Qualität Ihres ML-Tools einmal nicht gut ist, kehren Sie in dieses Tutorial zurück und überlegen Sie, wie Sie vielleicht die Ausgangsdaten "optimieren" können!